In [1]:
!pip install -q langgraph langchain-openai langchain-chroma langchain-huggingface sentence-transformers langchain-community

In [2]:
import os
import getpass
from typing import List, Dict
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [3]:


if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key를 입력하세요: ")

# 1. 임베딩 & DB 연결
hf_embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

vectorstore = Chroma(
    persist_directory="/content/drive/MyDrive/kt_cs_agent/chroma_db_kt_terms",
    embedding_function=hf_embeddings,
    collection_name="kt_terms"
)

# ====================================================
# 핵심 로직: 하이브리드 검색 함수 (Scoped 2개 + Global 1개)
# ====================================================
def hybrid_search_engine(inputs: Dict):
    target_doc = inputs["target_doc"].strip()
    query = inputs["search_query"].strip()

    print(f"🎯 [전략] '{target_doc}' 문서에서 2개, 전체에서 1개 검색합니다.")

    docs = []

    # 1. [Scoped Search] 추천 문서에서 2개 검색 (메타데이터 필터링)
    # 주의: 실제 DB에 저장된 'source' 메타데이터 값과 일치해야 필터링됨
    # 여기서는 파일명 일부가 포함되면 검색되도록 가정하거나, 정확한 파일명을 써야 함
    if target_doc != "없음":
        # Chroma는 metadata filter로 정확한 일치를 주로 사용함
        # 실제 구현 시엔 DB의 정확한 source 경로를 매핑해주는 로직이 필요할 수 있음(주소와 각 문서 이름을 db에 저장해서 불러오거나 딕셔너리형태로 따로 저장해도 좋을듯 하면 좋을 듯)
        # 여기서는 예시로 filter 기능을 보여줌
        scoped_results = vectorstore.similarity_search(
            query,
            k=2,
            # filter={"source": target_doc} # 실제 파일 경로와 일치해야 함!(벡터 db 생성시)
        )
        docs.extend(scoped_results)

    # 2. [Global Search] 제한 없이 1개 검색 (보험용)
    global_results = vectorstore.similarity_search(query, k=1)
    docs.extend(global_results)

    # 중복 제거 (문서 ID나 내용 기반)
    unique_docs = []
    seen = set()
    for doc in docs:
        if doc.page_content not in seen:
            unique_docs.append(doc)
            seen.add(doc.page_content)

    return unique_docs

# 포맷팅 함수
def format_docs(docs: List[Document]):
    output = ""
    for i, doc in enumerate(docs):
        source = doc.metadata.get("source", "Unknown").split("/")[-1]
        page = doc.metadata.get("page", 0) + 1
        output += f"\n📄 [{i+1}] {source} (p.{page})\n{doc.page_content[:100]}...\n"
    return output

# ====================================================
# LCEL 체인 구성
# ====================================================
llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

# 프롬프트: 문서 추천과 키워드를 구분자(|)로 나눠서 출력하도록 지시
prompt = ChatPromptTemplate.from_template(
    """
    당신은 상담 내용을 분석하여 '검색할 문서'와 '검색 키워드'를 결정하는 관리자입니다.

    [보유 문서 목록]
    - 인터넷서비스이용약관_202509.pdf
    - TV서비스이용약관.pdf (가상의 파일)
    - 모바일서비스이용약관.pdf (가상의 파일)

    [상담 요약]
    {summary}

    위 내용을 보고 가장 연관된 '문서 파일명(위 목록 중 택1)'과 '검색 키워드'를 아래 형식으로 출력하세요.
    문서를 특정하기 어려우면 문서명에 '없음'이라고 적으세요.

    형식: 문서명 | 키워드
    예시: 인터넷서비스이용약관_202509.pdf | 해지 위약금 산정식
    """
)

# 파서: LLM 출력을 잘라서 딕셔너리로 만듦
def parse_output(text: str):
    try:
        parts = text.split("|")
        return {"target_doc": parts[0].strip(), "search_query": parts[1].strip()}
    except:
        return {"target_doc": "없음", "search_query": text}

# 체인 연결
chain = (
    prompt
    | llm
    | StrOutputParser()
    | parse_output
    | RunnablePassthrough.assign(documents=hybrid_search_engine) # 검색 실행
)

# 실행
summary = "인터넷 약정이 남았는데 해지하면 위약금이 얼마나 나오나요?"
result = chain.invoke({"summary": summary})

print("\n" + "="*60)
print(f"🗣️ 상담 요약: {summary}")
print(f"🤖 AI 판단: 문서[{result['target_doc']}] / 키워드[{result['search_query']}]")
print("-" * 60)
print(format_docs(result['documents']))
print("=" * 60)

OpenAI API Key를 입력하세요: ··········


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


🎯 [전략] '인터넷서비스이용약관_202509.pdf' 문서에서 2개, 전체에서 1개 검색합니다.

🗣️ 상담 요약: 인터넷 약정이 남았는데 해지하면 위약금이 얼마나 나오나요?
🤖 AI 판단: 문서[인터넷서비스이용약관_202509.pdf] / 키워드[해지 위약금 산정식]
------------------------------------------------------------

📄 [1] (이용약관전문)인터넷서비스이용약관_202509.pdf (p.116)
※ 계약기간 이내 해지시는 할인받은 요금 및 단말장치사용료를 반환하여야 합니다. 
  - 이용료 산정식 = 할인금액 ⅹ 경과월수 ⅹ(1-사용기간 할인율/계약기간 할인율) 
   ·...

📄 [2] (이용약관전문)인터넷서비스이용약관_202509.pdf (p.167)
-  167 - 
 
※ 계약기간 이내 AP를 해지하거나 계약기간을 단축할 시는 추가 할인된 요금을 반환해야 하며, 
   할인반환금은 아래의 산식을 적용 받습니다. 
   - 산...

